# Sheikh WhatsApp Bot v8.0 Pro (Bank Al-Ma'rifa)
Includes Admin Authority Engine, Vision Photo Solver, PDF Summarizer, Interactive Quizzes & Leaderboard, Passive File Assist, Auto-Welcome, Catch-Up Summary, Missing-Course Tracker, Anti-Link & Anti-Spam.


In [ ]:
# ================= ONE-TAP "SHEIKH BOT" v8.0 PRO =================
PAIRING_PHONE_NUMBER = "" #@param {type:"string"}
OWNER_NUMBERS = "" #@param {type:"string"}
RESET_WHATSAPP_SESSION = False #@param {type:"boolean"}
GEMINI_API_KEY = "" #@param {type:"string"}
GEMINI_MODEL = "gemini-2.5-flash" #@param ["gemini-2.5-flash", "gemini-2.5-flash-lite", "gemini-2.0-flash"]
TARGET_GROUP_INVITE_URL = "https://chat.whatsapp.com/KWzAksUMH0j8HtIsRnJHjj?s=sh&p=a&ilr=1&amv=0" #@param {type:"string"}
ALLOW_PRIVATE_DM = True #@param {type:"boolean"}
ENABLE_PASSIVE_ASSIST = True #@param {type:"boolean"}
ENABLE_AUTO_WELCOME = True #@param {type:"boolean"}
ENABLE_ANTI_LINK = True #@param {type:"boolean"}
ENABLE_ANTI_SPAM = True #@param {type:"boolean"}
ENABLE_MODERATION_3_STRIKES = True #@param {type:"boolean"}
CHAT_COOLDOWN_MINUTES = 10 #@param {type:"integer"}
RANDOM_CHAT_CHANCE = 8 #@param {type:"slider", min:0, max:50, step:2}
VAULT_NAME = "\u0628\u0646\u0643 \u0627\u0644\u0645\u0639\u0631\u0641\u0629" #@param {type:"string"}
MAX_FILES_PER_REPLY = 4 #@param {type:"slider", min:1, max:8, step:1}
MAX_SEND_FILE_SIZE_MB = 75 #@param {type:"integer"}

import os
import shutil
import subprocess
import time
from google.colab import drive

drive.mount("/content/drive", force_remount=False)

bot_work_dir = "/content/sheikh_bot_v8"
os.makedirs(bot_work_dir, exist_ok=True)
drive_bot_dir = "/content/drive/MyDrive/Telegram_PDF_Archive/Sheikh_WhatsApp_Bot"
auth_dir = "/content/drive/MyDrive/Telegram_PDF_Archive/Logs_and_State/sheikh_whatsapp_auth"

if RESET_WHATSAPP_SESSION and os.path.exists(auth_dir):
    shutil.rmtree(auth_dir, ignore_errors=True)
    print("[Auth] Cleared previous session. Fresh 8-digit pairing code will be generated.")

# Install Node 22 + poppler-utils (for fast PDF summarization/quizzes)
node_ver = subprocess.run(["node", "-v"], capture_output=True, text=True)
if node_ver.returncode != 0 or not node_ver.stdout.startswith("v22"):
    print("Installing Node.js v22 + poppler-utils...")
    subprocess.run("curl -fsSL https://deb.nodesource.com/setup_22.x | bash - && apt-get install -y -qq nodejs poppler-utils", shell=True, check=False)
else:
    subprocess.run("apt-get install -y -qq poppler-utils", shell=True, check=False)

shutil.copy2(os.path.join(drive_bot_dir, "sheikh_whatsapp_bot_v8.js"), os.path.join(bot_work_dir, "sheikh_whatsapp_bot_v8.js"))
shutil.copy2(os.path.join(drive_bot_dir, "package_v8.json"), os.path.join(bot_work_dir, "package.json"))

print("Installing Baileys 7.0.0-rc14 + PDF Engine (~20s)...")
subprocess.check_call(["npm", "install", "--silent", "--no-audit", "--no-fund"], cwd=bot_work_dir)

resolved_vault_name = VAULT_NAME.encode("utf-8").decode("unicode_escape") if "\\u" in VAULT_NAME else VAULT_NAME

env = os.environ.copy()
env["PAIRING_PHONE_NUMBER"] = PAIRING_PHONE_NUMBER.strip().replace("+", "").replace(" ", "")
env["OWNER_NUMBERS"] = OWNER_NUMBERS.strip()
env["GEMINI_API_KEY"] = GEMINI_API_KEY.strip()
env["GEMINI_MODEL"] = GEMINI_MODEL.strip()
env["TARGET_GROUP_INVITE_URL"] = TARGET_GROUP_INVITE_URL.strip()
env["ALLOW_PRIVATE_DM"] = "true" if ALLOW_PRIVATE_DM else "false"
env["ENABLE_PASSIVE_ASSIST"] = "true" if ENABLE_PASSIVE_ASSIST else "false"
env["ENABLE_AUTO_WELCOME"] = "true" if ENABLE_AUTO_WELCOME else "false"
env["ENABLE_ANTI_LINK"] = "true" if ENABLE_ANTI_LINK else "false"
env["ENABLE_ANTI_SPAM"] = "true" if ENABLE_ANTI_SPAM else "false"
env["ENABLE_MODERATION"] = "true" if ENABLE_MODERATION_3_STRIKES else "false"
env["CHAT_COOLDOWN_MINUTES"] = str(CHAT_COOLDOWN_MINUTES)
env["RANDOM_CHAT_CHANCE"] = str(RANDOM_CHAT_CHANCE)
env["VAULT_NAME"] = resolved_vault_name.strip()
env["MAX_FILES_PER_REPLY"] = str(MAX_FILES_PER_REPLY)
env["MAX_SEND_FILE_SIZE_MB"] = str(MAX_SEND_FILE_SIZE_MB)
env["UNIQUE_PDFS_DIR"] = "/content/drive/MyDrive/Telegram_PDF_Archive/Unique_PDFs"
env["INDEX_CSV_PATH"] = "/content/drive/MyDrive/Telegram_PDF_Archive/Logs_and_State/whatsapp_bot_index.csv"
env["AUTH_DIR"] = auth_dir
env["STATE_DIR"] = "/content/drive/MyDrive/Telegram_PDF_Archive/Logs_and_State"
env["PORT"] = "3000"

print("==================================================================")
print("STARTING SHEIKH WHATSAPP BOT v8.0 PRO (WATCHDOG ACTIVE)...")
print("==================================================================")

while True:
    process = subprocess.Popen(
        ["node", "sheikh_whatsapp_bot_v8.js"],
        cwd=bot_work_dir,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    try:
        for line in process.stdout:
            print(line, end="", flush=True)
        code = process.wait()
        print(f"\n[Watchdog] Process exited ({code}). Restarting in 5s...")
        time.sleep(5)
    except KeyboardInterrupt:
        process.terminate()
        print("\nSheikh Bot v8.0 Pro stopped cleanly.")
        break
